# 01 - Data exploration

This shared notebook is the starting point for the intermediate delivery. It audits the supplied files before the group chooses preprocessing or modelling decisions. Run all cells from the repository root in JupyterLab.

**Question:** Can we predict whether a customer will make another purchase within 90 days of the current purchase?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
TARGET = 'repeat_purchase_90d'
DATA_DIR = Path('data')

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
clientes = pd.read_csv(DATA_DIR / 'clientes.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train.shape[0]:,} rows x {train.shape[1]} columns')
print(f'Test: {test.shape[0]:,} rows x {test.shape[1]} columns')
print(f'Clientes: {clientes.shape[0]:,} rows x {clientes.shape[1]} columns')
print(f'Sample submission: {sample_submission.shape[0]:,} rows x {sample_submission.shape[1]} columns')

## 1. First look

Each row of `train` and `test` is a purchase occasion. `train` additionally contains the known binary target. `clientes` has one customer-level row per `customer_id` and is optional extra information; do not merge it until we inspect its quality and test whether it improves validation performance.

In [ ]:
display(train.head())
display(clientes.head())
display(sample_submission.head())

In [ ]:
print('Target counts:')
display(train[TARGET].value_counts().rename_axis(TARGET).to_frame('count'))

print('Target proportions:')
display(train[TARGET].value_counts(normalize=True).rename_axis(TARGET).to_frame('proportion'))

ax = sns.countplot(data=train, x=TARGET)
ax.set_title('Target distribution in train.csv')
plt.show()

## 2. Structure and data quality

Inspect data types, descriptive statistics, missing values and duplicate rows. Record observations that lead to an actual modelling decision; do not include plots or checks with no purpose in the final clean notebook.

In [ ]:
train.info()
display(train.describe(include='all').T)

missing_train = (
    train.isna().sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
    .to_frame('missing_count')
)
missing_train['missing_pct'] = missing_train['missing_count'] / len(train)
display(missing_train)

print(f'Exact duplicate rows in train: {train.duplicated().sum()}')
print(f'Duplicate customer_id rows in clientes: {clientes.customer_id.duplicated().sum()}')

In [ ]:
train_features = set(train.columns) - {TARGET}
test_features = set(test.columns)

print('Features only in train:', sorted(train_features - test_features))
print('Features only in test:', sorted(test_features - train_features))
assert train_features == test_features, 'Train and test predictors do not match.'
assert sample_submission.columns.tolist() == ['ID', TARGET], 'Unexpected submission template.'

## 3. Initial feature roles

`ID` is the row identifier for Kaggle and must not be used as a predictor. `customer_id` is a join key and potentially identifier-like, so it requires an explicit experiment and justification before use. `purchase_date` must be converted to a date before making any time-based features or validation split.

In [ ]:
id_columns = ['ID', 'customer_id']
date_columns = ['purchase_date']

numeric_columns = train.select_dtypes(include='number').columns.drop(TARGET).tolist()
categorical_columns = [
    column for column in train.select_dtypes(include=['object', 'category']).columns
    if column not in id_columns + date_columns
]

print('Numeric predictors:', numeric_columns)
print('Categorical predictors:', categorical_columns)
print('Potential identifier/date columns:', id_columns + date_columns)

dates = pd.to_datetime(train['purchase_date'], errors='coerce')
print(f'Purchase-date range: {dates.min().date()} to {dates.max().date()}')
print(f'Invalid or missing purchase dates: {dates.isna().sum()}')

## Next decisions

Before building the first model, agree on a validation split that reflects the fact that `test.csv` contains later purchases. The next notebook should build a **minimal baseline pipeline**: numerical median imputation, categorical missing level plus one-hot encoding, no use of `ID`, and a Dummy Classifier plus two real models. Then test one change at a time against that baseline.